# Exploratory Data Analysis: Generative AI Adoption and Perception in Software Engineering

This notebook delivers a **deep, reproducible EDA** of survey datasets stored under `base_files/<cohort>/responses<stage>.csv`.

## Objective
Understand **adoption** and **perception** of generative AI in software engineering, with emphasis on:

- respondent profile and experience context
- overall and software-engineering-specific AI usage
- tools and activities associated with generative AI adoption
- perceived usefulness, productivity, quality, process change, and skill implications
- perceived impact on software engineering roles
- exploratory subgroup comparisons and light statistical testing

## Important notes
- Each loaded response receives two provenance columns: `cohort` and `stage`.
- The loader reads only files matching `responses<stage>.csv` directly under each cohort folder in `base_files`.
- Personally identifying columns are removed from the analytical dataset.
- Two duplicated fields with suffix `.1` are treated as import artifacts. The original versions are fully empty; the `.1` versions contain the actual responses.
- Open-ended text responses are **kept in the cleaned dataset** for completeness but **excluded from qualitative text analysis**, per project requirements.
- Statistical tests in this notebook are **exploratory**, not confirmatory, due to the small sample size.


In [7]:
# Core imports
from pathlib import Path
import re
import textwrap
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from scipy.stats import (
    spearmanr,
    mannwhitneyu,
    kruskal,
    chi2_contingency
)

# Reproducibility / display config
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

BASE_FILES_DIR = Path("..//base_files")
OUTPUT_DIR = Path("eda_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

STAGE_FILE_PATTERN = re.compile(r"^responses(\d+)\.csv$", re.IGNORECASE)


In [8]:
def extract_stage_from_filename(file_path):
    match = STAGE_FILE_PATTERN.match(file_path.name)
    if match:
        return int(match.group(1))
    return pd.NA


def load_survey_data(base_files_dir=BASE_FILES_DIR):
    if not base_files_dir.exists():
        raise FileNotFoundError(
            "The base_files directory was not found. Expected cohort data in "
            "'base_files/<cohort>/responses<stage>.csv'."
        )

    source_records = []
    frames = []

    cohort_dirs = sorted(
        [path for path in base_files_dir.iterdir() if path.is_dir()],
        key=lambda path: path.name
    )

    for cohort_dir in cohort_dirs:
        cohort = cohort_dir.name
        csv_files = sorted(
            [
                path for path in cohort_dir.iterdir()
                if path.is_file() and STAGE_FILE_PATTERN.match(path.name)
            ],
            key=lambda path: (extract_stage_from_filename(path), path.name)
        )

        for csv_path in csv_files:
            stage = extract_stage_from_filename(csv_path)
            frame = pd.read_csv(csv_path)
            frame["cohort"] = cohort
            frame["stage"] = stage
            frames.append(frame)

            source_records.append({
                "cohort": cohort,
                "stage": stage,
                "file_name": csv_path.name,
                "relative_path": str(csv_path.as_posix()),
                "n_rows": len(frame),
                "n_columns": frame.shape[1]
            })

    if not frames:
        raise FileNotFoundError(
            "No files matching 'responses<stage>.csv' were found under base_files/<cohort>."
        )

    raw = pd.concat(frames, ignore_index=True, sort=False)
    source_files = pd.DataFrame(source_records).sort_values(
        ["cohort", "stage", "file_name"]
    ).reset_index(drop=True)

    return raw, source_files


# Load the raw dataset(s)
raw, source_files = load_survey_data()

print(f"Loaded {len(source_files)} source file(s)")
display(source_files)

print(f"Raw shape: {raw.shape[0]} rows x {raw.shape[1]} columns")
display(raw.head(3))


Loaded 4 source file(s)


,cohort,stage,file_name,relative_path,n_rows,n_columns
0,2025.2,1,responses1.csv,../base_files/2025.2/responses1.csv,46,51
1,2025.2,2,responses2.csv,../base_files/2025.2/responses2.csv,41,48
2,2025.2,3,responses3.csv,../base_files/2025.2/responses3.csv,40,48
3,2026.1,1,responses1.csv,../base_files/2026.1/responses1.csv,22,49


Raw shape: 149 rows x 59 columns


,Timestamp,Email Address,Nome completo,Em qual semestre do curso de Engenharia de Software você está?,Você já possui outra graduação?,Qual é a sua faixa etária?,Qual é o seu gênero?,Você concluiu o Ensino Médio em escola pública?,Quantos anos de experiência em desenvolvimento de software você possui?,Você está empregado atualmente na área de software?,Quantos projetos de engenharia de software você participou ao longo de sua carreira?,Qual papel você mais se identifica em projetos de software?,Qual o número de membros da maior equipe de projeto em que você já atuou?,Com que frequência você utiliza ferramentas de IA generativa em seu dia a dia de maneira geral?,Com que frequência você utiliza ferramentas de IA Generativa especificamente em projetos de engenharia de software?,Para quais atividades você costuma utilizar ferramentas de IA generativa? (marque todas as que se aplicam),Quais ferramentas de IA generativa você utilizou nos últimos 12 meses em projetos de engenharia de software? (selecione todas que se aplicam),Qual metodologia de desenvolvimento de software você mais utilizou em seus projetos?,Avalie seu nível de concordância com as afirmações abaixo relacionadas ao uso de ferramentas de IA generativa em engenharia de software. [Ferramentas de IA generativa em Engenharia de Software são úteis.],Avalie seu nível de concordância com as afirmações abaixo relacionadas ao uso de ferramentas de IA generativa em engenharia de software. [IA generativa aumenta minha produtividade na engenharia de software.],Avalie seu nível de concordância com as afirmações abaixo relacionadas ao uso de ferramentas de IA generativa em engenharia de software. [IA generativa melhora a qualidade do software produzido.],Avalie seu nível de concordância com as afirmações abaixo relacionadas ao uso de ferramentas de IA generativa em engenharia de software. [O uso de IA generativa altera o fluxo de desenvolvimento de sistemas.],Avalie seu nível de concordância com as afirmações abaixo relacionadas ao uso de ferramentas de IA generativa em engenharia de software. [É possível utilizar qualquer metodologia de desenvolvimento de software com IA generativa.],Avalie seu nível de concordância com as afirmações abaixo relacionadas ao uso de ferramentas de IA generativa em engenharia de software. [Existe metodologia mais adequada às ferramentas de IA generativa.],Avalie seu nível de concordância com as afirmações abaixo relacionadas ao uso de ferramentas de IA generativa em engenharia de software. [O uso de ferramentas de IA generativa exige novas competências que não fazem parte da formação tradicional em Engenharia de Software.],"Avalie seu nível de concordância com as afirmações abaixo relacionadas ao uso de ferramentas de IA generativa em engenharia de software. [Novos papéis profissionais surgirão devido à IA generativa (por exemplo, AI Prompt Engineer, AI Auditor).]",Avalie seu nível de concordância com as afirmações abaixo relacionadas ao uso de ferramentas de IA generativa em engenharia de software. [Sinto-me confiante em revisar e integrar código gerado por IA generativa.],A função de backend será extinta ou severamente afetada pelo uso de ferramentas de IA generativa na engenharia de software. Indique seu grau de concordância de 1 (Discordo totalmente) a 5 (Concordo totalmente).,A função de frontend será extinta ou severamente afetada pelo uso de ferramentas de IA generativa na engenharia de software. Indique seu grau de concordância de 1 (Discordo totalmente) a 5 (Concordo totalmente).,A função de QA (Garantia de Qualidade) será extinta ou severamente afetada pelo uso de ferramentas de IA generativa na engenharia de software. Indique seu grau de concordância de 1 (Discordo totalmente) a 5 (Concordo totalmente).,A função de gerente de projeto será extinta ou severamente afetada pelo uso de ferramentas de IA generativa na engenharia de software. Indique seu grau de concordância de 1 (Discordo totalmente) a 5 (Concordo totalmente).,A função de Product 

## 1. Initial schema audit

This section checks:
- dataset size
- column names
- duplicated fields introduced by import/export
- obvious missingness issues


In [9]:
schema = pd.DataFrame({
    "column": raw.columns,
    "dtype": raw.dtypes.astype(str).values,
    "non_null": raw.notna().sum().values,
    "missing": raw.isna().sum().values,
    "missing_pct": (raw.isna().mean().values * 100).round(2)
})
display(schema)


,column,dtype,non_null,missing,missing_pct
0,Timestamp,str,149,0,0.00
1,Email Address,str,149,0,0.00
2,Nome completo,str,68,81,54.36
3,Em qual semestre do curso de Engenharia de Sof...,str,68,81,54.36
4,Você já possui outra graduação?,str,149,0,0.00
5,Qual é a sua faixa etária?,str,149,0,0.00
6,Qual é o seu gênero?,str,149,0,0.00
7,Você concluiu o Ensino Médio em escola pública?,str,149,0,0.00
8,Quantos anos de experiência em desenvolvimento...,str,149,0,0.00
9,Você está empregado atualmente na área de soft...,str,149,0,0.00


In [10]:
# Identify import-style duplicated columns, such as 'question' and 'question.1'
duplicate_like_cols = [c for c in raw.columns if c.endswith(".1")]
duplicate_audit = []

for dup_col in duplicate_like_cols:
    base_col = dup_col[:-2]
    base_exists = base_col in raw.columns
    duplicate_audit.append({
        "base_column": base_col,
        "duplicate_column": dup_col,
        "base_exists": base_exists,
        "base_non_null": int(raw[base_col].notna().sum()) if base_exists else np.nan,
        "duplicate_non_null": int(raw[dup_col].notna().sum()),
        "base_all_missing": bool(raw[base_col].isna().all()) if base_exists else np.nan
    })

duplicate_audit = pd.DataFrame(duplicate_audit)
display(duplicate_audit)

if not duplicate_audit.empty:
    print("Interpretation:")
    for _, row in duplicate_audit.iterrows():
        print(
            f"- Base column all missing? {row['base_all_missing']} | "
            f"Using populated duplicate: {row['duplicate_column']}"
        )


,base_column,duplicate_column,base_exists,base_non_null,duplicate_non_null,base_all_missing
0,Qual foi o maior benefício que você obteve (ou...,Qual foi o maior benefício que você obteve (ou...,True,103,46,False
1,"Em 5 anos, como você enxerga o impacto das fer...","Em 5 anos, como você enxerga o impacto das fer...",True,103,46,False


Interpretation:
- Base column all missing? False | Using populated duplicate: Qual foi o maior benefício que você obteve (ou imagina) ao usar ferramentas de IA generativa em projetos de Engenharia de Software?.1
- Base column all missing? False | Using populated duplicate: Em 5 anos, como você enxerga o impacto das ferramentas de IA generativa em sua carreira de engenharia de software?.1
